# 02 · Construir la serie mensual con cobertura y media móvil · Resolución

Solución comentada de esta práctica de livecoding.

Seleccioná el kernel de `.venv` y usá **Reiniciar kernel y ejecutar todo**. La sección 3 contiene la función resuelta y la sección 4 ejecuta las pruebas de la práctica junto con la comprobación sobre datos reales. El resultado completo queda en la variable `resultado`.

La carga reutiliza `practicas_livecoding/preparacion.py` y los datasets ya descargados en `practicas_livecoding/datos/`. Conservá ambas carpetas dentro del mismo proyecto.

## 1. Preparación lista

La base equivale a `base` del notebook original: fecha como índice, nombres preparados y faltantes, negativos y ceros conservados.

Los archivos de `practicas_livecoding/datos/` ya están descargados. Si faltan, se descargan una vez desde Kaggle y se guardan localmente.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from pandas.testing import assert_frame_equal

# Encuentra la carpeta tanto desde la raíz del proyecto como desde este notebook.
ubicaciones = [Path.cwd(), *Path.cwd().parents]
carpeta = next(
    (
        candidata
        for ubicacion in ubicaciones
        for candidata in [ubicacion, ubicacion / "practicas_livecoding"]
        if candidata.name == "practicas_livecoding"
        and (candidata / "preparacion.py").is_file()
    ),
    None,
)
if carpeta is None:
    raise FileNotFoundError("Abrí el notebook dentro de la carpeta practicas_livecoding.")

if str(carpeta) not in sys.path:
    sys.path.insert(0, str(carpeta))

from preparacion import cargar_viviendas, cargar_energia

pd.set_option("display.max_columns", 20)

In [2]:
datos = cargar_energia()

print(f"Filas: {len(datos):,} | Columnas: {len(datos.columns)}")
display(datos.head())

Filas: 285,499 | Columnas: 13


,TipoConsumidor,Sistema,UF,Classe,Setor Econômico - N2,Setor Econômico - N3,Tipo Tensão - N1,Tipo Tensão - N2,Tipo Tensão - N3,Faixa de Consumo N1,Faixa de Consumo N2,Consumidores,Consumo
Data,,,,,,,,,,,,,
2014-01-01,Cativo,Sudeste / Centro-Oeste,RO,Residencial,Convencional (Excepto Baixa Renda),TOTAL,A - Alta Tensão,TOTAL,TOTAL,Alta Tensão,Alta Tensão,1.0,8.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,SP,Rural,Serviço Público de Irrigação,TOTAL,A - Alta Tensão,"A-4 - 2,3 a 25 kV",TOTAL,Não Aplicável,Não Aplicável,6.0,145.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,SP,Rural,Serviço Público de Irrigação,TOTAL,B - Baixa Tensão,TOTAL,TOTAL,Não Aplicável,Não Aplicável,31.0,18.0
2014-01-01,Cativo,Sudeste / Centro-Oeste,MS,Residencial,Baixa Renda,Baixa Renda N2,B - Baixa Tensão,TOTAL,TOTAL,Baixa Renda,> 200 kWh,44193.0,13171.0
2014-01-01,Cativo,Sistemas Isolados,AC,Residencial,Convencional (Excepto Baixa Renda),TOTAL,B - Baixa Tensão,TOTAL,TOTAL,Convencional,0-30 kWh,5216.0,149.0


## 2. Consigna y resultado esperado

A partir de la base completa con índice `Data`, construí una tabla mensual con
frecuencia de inicio de mes (`MS`) e índice llamado `Data`. Incluí todos los meses
entre la primera y la última fecha, aunque alguno no tenga registros.

| Columna, en este orden | Cálculo |
|---|---|
| `consumo_total` | Suma de `Consumo` con `min_count=1` |
| `consumidores_total` | Suma de `Consumidores` con `min_count=1` |
| `registros` | Cantidad de filas del mes, incluidos los faltantes |
| `consumo_presente` | Cantidad de valores de `Consumo` no nulos |
| `cobertura_pct` | `consumo_presente / registros * 100` |
| `media_movil_12` | Media de `consumo_total` en los últimos 12 meses, incluido el actual |

La media móvil exige 12 valores mensuales válidos: usá `min_periods=12`, sin centrar.
Un mes sin mediciones no se convierte en cero. Un mes sin filas tiene ambos conteos en cero
y cobertura `NaN`. Conservá los negativos. No modifiques la entrada; podés asumir que no está vacía.

## 3. Resolución comentada

1. `resample('MS')` agrupa los registros por inicio de mes e incorpora los meses intermedios.
2. `sum(min_count=1)` mantiene `NaN` cuando no hay ninguna medición para sumar.
3. La cobertura compara mediciones presentes (`count`) con filas declaradas (`size`).
4. `rolling(12, min_periods=12)` solo produce una media si hay doce totales válidos en la ventana.

Un mes sin filas tiene cobertura `NaN` por el cociente `0 / 0`.
La media móvil usa el mes actual y los anteriores; no incorpora meses futuros.

In [3]:
def resumir_consumo_mensual(datos):
    """Calcula los totales, la cobertura y una media móvil de doce meses."""
    # min_count=1 evita que un mes sin mediciones aparezca como consumo cero.
    mensual = datos[["Consumo", "Consumidores"]].resample("MS").sum(min_count=1)
    mensual = mensual.rename(columns={
        "Consumo": "consumo_total",
        "Consumidores": "consumidores_total",
    })

    # size cuenta todas las filas; count cuenta solo las mediciones presentes.
    mensual["registros"] = datos["Consumo"].resample("MS").size()
    mensual["consumo_presente"] = datos["Consumo"].resample("MS").count()
    mensual["cobertura_pct"] = (
        mensual["consumo_presente"] / mensual["registros"] * 100
    )

    # La ventana incluye el mes actual y los once anteriores; exige doce datos.
    mensual["media_movil_12"] = (
        mensual["consumo_total"].rolling(12, min_periods=12).mean()
    )
    return mensual

## 4. Pruebas y ejecución real

Se conservan los casos y las comprobaciones de la práctica original. Los mensajes `OK` confirman que pasan y que la entrada permanece intacta. La tabla muestra las primeras 20 filas; `resultado` contiene la salida completa.

In [4]:
# Estas pruebas no reemplazan tu función: comprueban su resultado.
def comparar(actual, esperado):
    assert isinstance(actual, pd.DataFrame), "La función debe devolver un DataFrame."
    assert_frame_equal(
        actual, esperado,
        check_dtype=False, check_index_type=False, check_column_type=False,
        check_freq=False, rtol=1e-6, atol=1e-8,
    )


def probar(funcion):
    fechas = pd.date_range("2023-01-01", periods=14, freq="MS", name="Data")
    caso = pd.DataFrame({
        "Consumo": [10, np.nan, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140],
        "Consumidores": [1, np.nan, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
    }, index=fechas)
    extra = pd.DataFrame(
        {"Consumo": [5], "Consumidores": [1]},
        index=pd.DatetimeIndex(["2023-01-01"], name="Data"),
    )
    caso = pd.concat([caso, extra])
    original = caso.copy(deep=True)
    esperado = pd.DataFrame({
        "consumo_total": [15, np.nan, 30, 40, 50, 60, 70, 80, 90, 100, 110, 120, 130, 140],
        "consumidores_total": [2, np.nan, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14],
        "registros": [2] + [1] * 13,
        "consumo_presente": [2, 0] + [1] * 12,
        "cobertura_pct": [100, 0] + [100] * 12,
        "media_movil_12": [np.nan] * 13 + [85],
    }, index=fechas)
    comparar(funcion(caso), esperado)

    # Febrero no tiene filas; enero tiene un ajuste negativo.
    con_hueco = caso.iloc[[0, 2]].copy()
    con_hueco.iloc[0, 0] = -10
    esperado_hueco = pd.DataFrame({
        "consumo_total": [-10, np.nan, 30],
        "consumidores_total": [1, np.nan, 3],
        "registros": [1, 0, 1],
        "consumo_presente": [1, 0, 1],
        "cobertura_pct": [100, np.nan, 100],
        "media_movil_12": [np.nan] * 3,
    }, index=fechas[:3])
    comparar(funcion(con_hueco), esperado_hueco)
    comparar(caso, original)
    print("OK: meses repetidos, mes ausente, faltantes, negativos y ventana de 12 meses.")


probar(resumir_consumo_mensual)

# Comprobamos también que el dataset real permanezca intacto.
antes = datos.copy(deep=True)
resultado = resumir_consumo_mensual(datos)
comparar(datos, antes)
assert resultado.index.is_monotonic_increasing and resultado.index.is_unique
assert resultado["registros"].sum() == len(datos)
assert resultado["consumo_presente"].sum() == datos["Consumo"].notna().sum()
np.testing.assert_allclose(resultado["consumo_total"].sum(), datos["Consumo"].sum())
np.testing.assert_allclose(resultado["consumidores_total"].sum(), datos["Consumidores"].sum())
assert resultado["media_movil_12"].iloc[:11].isna().all()
print(f"OK: dataset real. Resultado: {len(resultado):,} filas.")
display(resultado.head(20))

OK: meses repetidos, mes ausente, faltantes, negativos y ventana de 12 meses.
OK: dataset real. Resultado: 120 filas.


,consumo_total,consumidores_total,registros,consumo_presente,cobertura_pct,media_movil_12
Data,,,,,,
2014-01-01,4.027976e+07,75036817.0,2134,2053,96.204311,NaN
2014-02-01,4.165349e+07,75282897.0,2136,2047,95.833333,NaN
2014-03-01,4.031539e+07,75468866.0,2135,2053,96.159251,NaN
2014-04-01,3.955025e+07,75650388.0,2136,2053,96.114232,NaN
2014-05-01,3.910050e+07,75858810.0,2141,2049,95.702943,NaN
2014-06-01,3.772647e+07,76014514.0,2136,2048,95.880150,NaN
2014-07-01,3.786681e+07,76213682.0,2143,2052,95.753616,NaN
2014-08-01,3.855069e+07,76321200.0,2143,2055,95.893607,NaN
2014-09-01,3.889498e+07,76622551.0,2142,2060,96.171802,NaN


## Para repasar

Volvé a la práctica sin resolver e intentá escribir la función desde cero. Explicá qué representa cada fila de la salida, cómo se resuelven los casos límite y por qué el cálculo respeta la consigna.